# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
# TODO
# GridSearchCV를 수행하세요.


rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}

grid = GridSearchCV(
    estimator=rf_model,
    param_grid=grid_params,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True
)

# 탐색 시간 측정
start_time = time.perf_counter()

grid.fit(X_train, y_train)

end_time = time.perf_counter()

elapsed_time = end_time - start_time

print(f"탐색 시간: {elapsed_time:.2f}초")
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

# 최적 조합의 cv_results_ 확인
best_idx = grid.best_index_

print("Mean Test Score:",
      (grid.cv_results_["mean_test_score"][best_idx]).round(4))

print("Std Test Score:",
      (grid.cv_results_["std_test_score"][best_idx]).round(4))


탐색 시간: 7.02초
Best Params: {'max_depth': 12, 'n_estimators': 200}
Best Score: 0.8206179847869421
Mean Test Score: 0.8206
Std Test Score: 0.0644


- 최고 CV 평균
    - 0.82
    - max_depth(12)와 n_estimators(200)의 조합으로 5개의 검증 R2을 평균한 값
    

- CV 표준편차
    - 0.06
    - 최적 조합 Fold별로 검증 점수가 평균 주변에서 얼마나 달랐는지를 나타냄

---

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  

- 장점
    - 지정한 조합을 빠짐없이 평가하여 하이퍼파라미터 튜닝 최적화를 용이하게 찾음
- 단점
    - 후보 수가 많아지면 탐색비용이 매우 느려짐


## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [3]:
# TODO
# RandomizedSearchCV를 수행하세요.

random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}

search = RandomizedSearchCV(
    rf_model,
    param_distributions=random_params,
    scoring='r2',
    n_iter=9,
    cv=5,
    random_state=42
    )

# 탐색 시간 측정
start_time = time.perf_counter()

search.fit(X_train, y_train)

end_time = time.perf_counter()

elapsed_time = end_time - start_time

print(f"탐색 시간: {elapsed_time:.2f}초")
print("Best Params:", search.best_params_)
print("Best Score:", search.best_score_)

# 최적 조합의 cv_results_ 확인

best_idx = search.best_index_

print("Mean Test Score:",
      (search.cv_results_["mean_test_score"][best_idx]).round(4))

print("Std Test Score:",
      (search.cv_results_["std_test_score"][best_idx]).round(4))



탐색 시간: 7.86초
Best Params: {'max_depth': 11, 'n_estimators': 230}
Best Score: 0.8210296383331107
Mean Test Score: 0.821
Std Test Score: 0.0651


> **해석**   
> Grid 서치의 CV 평균보다 약 0.0004 높다.   
> 반면에 표준편차는 0.06 정도로 커졌다.   
> Grid 서치와 지금 모델을 비교하여보면 평균 R2은 높지만 표준편차(변동)가 조금 더 크다.

---

### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**

- 내가 어떤 값을 넣어야 할지 모르고
- 모든 조합을 평가할 시간이 없을 때 주로 유용하다.

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [4]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.


# 1. objective 함수
def objective(trial):

    # 2. 하이퍼파라미터 탐색 범위
    n_estimators = trial.suggest_int("n_estimators", 100, 300)

    max_depth = trial.suggest_int("max_depth", 4, 12)

    # 3. RandomForestRegressor
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )

    # 4. 5-fold CV, R²
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    # 5. CV 평균 점수 반환
    return scores.mean()


# 6. TPESampler
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=3)

# 7. Study 생성
study = optuna.create_study(direction="maximize", sampler=sampler)

# 9. 탐색 시간 측정
start_time = time.perf_counter()

# 8. 9번 탐색
study.optimize(objective, n_trials=9)

end_time = time.perf_counter()

search_time = end_time - start_time


# 10. 결과 출력
print("Best Params:", study.best_params)
print("Best Value:", study.best_value)
print(f"Search Time: {search_time:.4f} sec")

[I 2026-10-10 22:29:14,284] A new study created in memory with name: no-name-bf70515d-7b2b-45b6-ae60-bae39b495c3e
[I 2026-10-10 22:29:15,001] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-10 22:29:15,946] Trial 1 finished with value: 0.8184595590761907 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-10 22:29:16,407] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-10 22:29:16,894] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-10 22:29:17,802] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

Best Params: {'n_estimators': 175, 'max_depth': 12}
Best Value: 0.8212542415716282
Search Time: 6.8974 sec


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  

- `Grid`는 정해 둔 모든 후보 조합을 평가하고, 
- `Random`은 정해진 횟수만큼 후보를 무작위로 선택
- `Optuna`는 초기 3개 trial 이후 결과를 참고하여 상대적으로 유리한 영역을 더 깊게 탐색하는 방식을 사용

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [5]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.

from sklearn.model_selection import cross_val_score, KFold

# 1. Optuna 최적 파라미터로 모델 생성
optuna_model = RandomForestRegressor(
    **study.best_params,
    random_state=42
)

# 2. 5-Fold 교차검증
cv = KFold(n_splits=5, shuffle=True, random_state=42)

optuna_scores = cross_val_score(
    optuna_model,
    X_train,
    y_train,
    cv=cv,
    scoring='r2',
    n_jobs=-1
)

# 3. CV 평균과 표준편차 계산
optuna_cv_mean = optuna_scores.mean()
optuna_cv_std = optuna_scores.std()

print('optuna CV 평균', optuna_cv_mean)
print('optuna CV 표준편차', optuna_cv_std)

optuna CV 평균 0.8217194140425228
optuna CV 표준편차 0.03887257110142188


- 앞서 진행한 세 가지 모델 결과 비교

    - `Grid`의 표준편차는 약 0.064425. `Random`은 0.06508였고, `Optuna`는 0.0642였다.
    - `Optuna`를 통해 결과를 구했을 때 평균이 가장 높고, 표준편차가 가장 작았다.

---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [6]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.

import joblib
import json
from sklearn.model_selection import cross_validate

# =========================
# 1. 세 가지 탐색 방법 비교
# =========================
# 실행한 탐색 결과에서 시간 가져오기
grid_time = grid.cv_results_['mean_fit_time'].sum() * 5     # 5-fold이므로 
random_time = search.cv_results_['mean_fit_time'].sum() * 5 
optuna_time = search_time

# Optuna도 같은 5-Fold로 비교
optuna_scores2 = cross_val_score(
    RandomForestRegressor(**study.best_params, random_state=42),
    X_train, y_train, cv=5, scoring='r2'
)

search_comparison = pd.DataFrame({
    'search_method': ['Grid', 'Random', 'Optuna'],

    'best_params': [
        grid.best_params_,
        search.best_params_,
        study.best_params
    ],

    'cv_mean': [
        grid.best_score_,
        search.best_score_,
        optuna_scores2.mean()
    ],

    'cv_std': [
        grid.cv_results_['std_test_score'][grid.best_index_],
        search.cv_results_['std_test_score'][search.best_index_],
        optuna_scores2.std()
    ],

    'search_seconds': [
        grid_time,
        random_time,
        optuna_time
    ]
})

display(search_comparison)

# CV 평균이 가장 높은 모델 선택
best = search_comparison.loc[search_comparison['cv_mean'].idxmax()]

print("최적 탐색 방법:", best['search_method'])
print("최적 파라미터:", best['best_params'])

# =========================
# 2. 튜닝 전후 성능 비교
# =========================
from sklearn.model_selection import cross_validate

# 1. 튜닝 전 모델
baseline = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42,  n_jobs=-1)

# 2. 튜닝 후 모델
tuned = RandomForestRegressor(**best['best_params'], random_state=42, n_jobs=-1)

# 3. 교차검증
before = cross_validate(baseline, X_train, y_train,  cv=5, scoring='r2', return_train_score=True)

after = cross_validate(tuned, X_train, y_train, cv=5, scoring='r2', return_train_score=True)

# 4. Fold별 비교
fold_comparison = pd.DataFrame({
    'fold': range(1, 6),
    'before_train': before['train_score'],
    'before_valid': before['test_score'],
    'after_train': after['train_score'],
    'after_valid': after['test_score']
})

display(fold_comparison)

# 5. 평균과 표준편차 비교
tuning_comparison = pd.DataFrame({
    'model': ['Baseline', 'Tuned'],
    'train_mean': [
        before['train_score'].mean(),
        after['train_score'].mean()
    ],
    'cv_mean': [
        before['test_score'].mean(),
        after['test_score'].mean()
    ],
    'cv_std': [
        before['test_score'].std(),
        after['test_score'].std()
    ]
})

tuning_comparison['train_valid_gap'] = tuning_comparison['train_mean'] - tuning_comparison['cv_mean']

display(tuning_comparison)

# 6. 성능 차이
r2_diff = after['test_score'].mean() - before['test_score'].mean()

print("튜닝 전후 R2 차이:", round(r2_diff, 4))

# ================================
# 3. 최종 모델 선택 및 Test 평가
# ================================
# 1. 더 좋은 모델 선택
if r2_diff > 0:
    final_model = tuned
    final_cv = after
    final_method = best['search_method']
else:
    final_model = baseline
    final_cv = before
    final_method = 'Baseline'

print("최종 선택:", final_method)

# 2. 최종 모델 학습
final_model.fit(X_train, y_train)

# 3. Test 성능 확인 (한 번만)
test_r2 = final_model.score(X_test, y_test)

cv_mean = final_cv['test_score'].mean()
cv_std = final_cv['test_score'].std()
cv_test_gap = cv_mean - test_r2

print("CV 평균 R2:", round(cv_mean, 4))
print("Test R2:", round(test_r2, 4))
print("CV와 Test 차이:", round(cv_test_gap, 4))


# =========================
# 4. 최종 모델 및 정보 저장
# =========================
# 1. 폴더 생성
os.makedirs('model_artifacts', exist_ok=True)

# 2. 모델 저장
joblib.dump(final_model, 'model_artifacts/best_model.pkl')

# 3. 모델 정보
model_meta = {
    'model_name': 'RandomForestRegressor',
    'saved_at': datetime.now().isoformat(),
    'selected_search_method': str(final_method),

    'best_params': {
        'n_estimators': int(final_model.n_estimators),
        'max_depth': (
            None if final_model.max_depth is None
            else int(final_model.max_depth)
        )
    },

    'cv_mean': float(cv_mean),
    'cv_std': float(cv_std),
    'test_r2': float(test_r2),

    'baseline_settings': {
        'n_estimators': 100,
        'max_depth': None
    },

    'tuned_candidate_settings': {
        'n_estimators': int(tuned.n_estimators),
        'max_depth': int(tuned.max_depth)
    },

    'tuned_candidate_search_method': str(best['search_method']),
    'tuning_cv_comparison': tuning_comparison.to_dict('records'),
    'tuning_cv_r2_delta': float(r2_diff),
    'cv_mean_minus_test_r2': float(cv_test_gap)
}

# 4. JSON 저장
with open('model_artifacts/model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=4)

print("모델 저장 완료!")

,search_method,best_params,cv_mean,cv_std,search_seconds
0,Grid,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,26.879449
1,Random,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,6.519431
2,Optuna,"{'n_estimators': 175, 'max_depth': 12}",0.821254,0.064281,6.897405


최적 탐색 방법: Optuna
최적 파라미터: {'n_estimators': 175, 'max_depth': 12}


,fold,before_train,before_valid,after_train,after_valid
0,1,0.971728,0.862667,0.971451,0.869414
1,2,0.975444,0.680790,0.976382,0.696696
2,3,0.974969,0.834270,0.972136,0.827994
3,4,0.974586,0.845045,0.972621,0.842770
4,5,0.971663,0.872282,0.970872,0.869396


,model,train_mean,cv_mean,cv_std,train_valid_gap
0,Baseline,0.973678,0.819011,0.070370,0.154667
1,Tuned,0.972692,0.821254,0.064281,0.151438


튜닝 전후 R2 차이: 0.0022
최종 선택: Optuna
CV 평균 R2: 0.8213
Test R2: 0.8911
CV와 Test 차이: -0.0698
모델 저장 완료!


### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  

- 아니요

- 성능이 높더라도 탐색시간이 몇 배 더 길다면 실제 운영과정에서 효율성이 떨어질 수 있기 때문에
- CV 표준편차, 모델의 안정성, 탐색과 학습 비용, 실제 운영여부를 같이 판단해야한다.


### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  

- Test 데이터는 학습하지 않은 새로운 데이터에 대해 모델의 성능을 판단하기 위해 사용한다.

- Test 데이터를 여러번 사용할 경우 해당 데이터에 맞춰 모델을 판단하기 때문에 일반화하기 어려워질 가능성이 있다.
- 따라서 마지막에 일반화하기 위한 자료로 사용해야 한다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**

> `CV 평균 R²`

  - 튜닝 전: 0.8190
  - 튜닝 후: 0.8213
  - 차이: 약 +0.0022

> `CV 표준편차`

  - 튜닝 전: 0.0704
  - 튜닝 후: 0.0643

> `Train–검증 점수 차이`

  - 튜닝 전: 0.1547
  - 튜닝 후: 0.1514


- 튜닝 후 CV 평균 R²이 약 0.0022 증가하였고, 표준편차도 감소하였다.

- Train과 검증 성능의 차이도 감소하였다.



### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**

> CV 평균 R²: 0.8213   
> 최종 Test R²: 0.8911   
> CV와 Test 차이: -0.0698   

- 최종 Test R²가 CV 평균보다 약 0.0698 높게 나타났다.
- 최종 모델이 학습에 사용하지 않은 Test 데이터에서도 비교적 좋은 예측 성능을 나타낸 것으로 보인다.
- 하지만 Train R²가 약 0.9727, CV 평균은 0.8213으로 나타나 과적합 가능성이 존재한다.

- 그리고 Test 결과는 한 번의 데이터 분할에서 나온 값이므로, 다른 분할 성능에서도 비슷한 결과가 나온다는 것을 확인 할 수 없기 때문에
- 추가적인 성능 안정성을 확인할 필요가 있다.


---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
    - Grid : 주어진 모든 후보 조합 평가
    - Random : n_iter만큼 무작위 조합 평가

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
    - Random Search 가 평가할 조합의 개수, 탐색 예산을 의미

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
    - `study` : 전체 탐색
    - `trial` : 한 번의 시도

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
    - **CV 평균**
    - **CV 표준편차**
    - 탐색 시간(학습시간)
    - 탐색 학습 비용

5. hold-out Test는 언제 사용해야 하나요?
    - 모든 탐색과 모델 선택이 끝난 뒤 마지막에 한 번 사용한다. 


6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?
    - 나중에 같은 모델을 재사용하고,
    - 어떤 조건으로 만들어졌는지 추적(Log)하여 재현성과 운영 관리를 높이기 위해 저장한다.